# Larval Brain vs. VNC: sex-aggregated DESeq2 analysis

Load the workbook's sample metadata and precomputed result tables, then focus on the Brain-versus-VNC tissue effect across sexes. The workbook contains sex-stratified DESeq2 contrasts, not raw counts or a pooled-sex tissue contrast; those existing results cannot be combined to create a valid pooled DESeq2 result.

## Workbook data and analysis scope

The sample design includes sex and treatment (control or SI). The code below loads the available result tables with appropriate data types and checks whether a pooled-sex tissue contrast or raw count matrix is present. A valid pooled-sex DESeq2 fit requires raw gene counts; a sensible main-effect model that keeps treatment adjusted would be `~ treatment + tissue`, with sex omitted.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

WORKBOOK_NAME = "differential_treatment_v0.0.0-21-g20af064_altered (2).xlsx"
workbook_candidates = [
    Path.cwd() / "data" / WORKBOOK_NAME,
    Path.cwd().parent / "data" / WORKBOOK_NAME,
]
WORKBOOK_PATH = next((path for path in workbook_candidates if path.is_file()), None)
if WORKBOOK_PATH is None:
    raise FileNotFoundError(
        f"Could not find {WORKBOOK_NAME!r}. Checked: {workbook_candidates}"
    )

workbook = pd.ExcelFile(WORKBOOK_PATH)
print(f"Workbook: {WORKBOOK_PATH}")
print(f"Sheets ({len(workbook.sheet_names)}): {workbook.sheet_names}")

Workbook: /Users/meryv/repos/DESeq2_L3/data/differential_treatment_v0.0.0-21-g20af064_altered (2).xlsx
Sheets (39): ['Parameters', 'Design', 'Class Sizes', 'Comparison Key', "(C1) 'SI - control|male', mix", "(C1) 'SI - control|female', mix", 'A', "(C2) 'female - male|SI', mix", 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'AA', 'AB', 'AC', 'AD', 'R Packages', 'R Details']


In [2]:
parameters = pd.read_excel(workbook, sheet_name="Parameters")
design = pd.read_excel(workbook, sheet_name="Design")
class_sizes = pd.read_excel(workbook, sheet_name="Class Sizes")
comparison_key = pd.read_excel(workbook, sheet_name="Comparison Key")

for column in ["stage", "sex", "treatment", "tissue", "mix", "sex_trt_tissue"]:
    design[column] = design[column].astype("category")
influential_values = design[".influential"].astype("string").str.strip().str.lower()
if influential_values.isin(["true", "false"]).all():
    design[".influential"] = influential_values.map({"true": True, "false": False}).astype("boolean")
else:
    raise ValueError("Unexpected values in the Design '.influential' column.")

for column in ["Design", "Comparison", "Group"]:
    class_sizes[column] = class_sizes[column].astype("category")
for column in ["Significant", "Total"]:
    class_sizes[column] = pd.to_numeric(class_sizes[column], errors="raise").astype("Int64")
comparison_key["Key"] = comparison_key["Key"].astype("string")
comparison_key["Comparison"] = comparison_key["Comparison"].astype("string")

print("Sample metadata:", design.shape)
display(design.dtypes.rename("dtype").to_frame())
display(design.groupby(["treatment", "tissue"], observed=True).size().unstack(fill_value=0))
display(parameters)

Sample metadata: (24, 8)


,dtype
ID,object
stage,category
sex,category
treatment,category
tissue,category
mix,category
sex_trt_tissue,category
.influential,boolean


tissue,VNC,brain
treatment,,
SI,6,6
control,6,6


,id,description
0,seed,A random seed of 1 is used to ensure reproduci...
1,spec,Using analysis plan 'principal'.
2,alignment,Using alignment settings 'BDGP6'
3,alpha,The threshold of statistical significance is 0.05
4,lfcThreshold,The threshold for absolute effect size is 0
5,filterFun,Default independent filtering
6,LRT_effect,"Use the '""default""' summary of an LRT 'effect ..."
7,gene_clust,bluster::HclustParam(cut.params = list(k = 12))
8,sample_clust,The value of sample_clust is bluster::HclustPa...


In [3]:
numeric_result_columns = [
    "baseMean", "log2FoldChange", "lfcSE", "stat", "pvalue", "entrez", "shrunkLFC"
]
result_tables = {}
for key in comparison_key["Key"]:
    result = pd.read_excel(workbook, sheet_name=str(key))
    result["id"] = result["id"].astype("string")
    for column in ["symbol", "sig", "class"]:
        result[column] = result[column].astype("string")
    for column in numeric_result_columns:
        result[column] = pd.to_numeric(result[column], errors="coerce")
    result["sig"] = result["sig"].astype("category")
    result["class"] = result["class"].astype("category")
    result_tables[str(key)] = result

print(f"Loaded {len(result_tables)} DESeq2 result tables.")
example_key = str(comparison_key.iloc[0]["Key"])
display(result_tables[example_key].dtypes.rename("dtype").to_frame())
display(comparison_key)

Loaded 30 DESeq2 result tables.


,dtype
id,string[python]
baseMean,float64
log2FoldChange,float64
lfcSE,float64
stat,float64
pvalue,float64
symbol,string[python]
entrez,float64
sig,category
shrunkLFC,float64


,Key,Comparison
0,A,"(C2) 'female - male|control', mix"
1,B,"(C3) 'SI - control|female - male', mix"
2,C,(C4) 'control_female_brain - control_male_brai...
3,D,(C4) 'control_female_VNC - control_male_brain'...
4,E,(C4) 'control_female_VNC - control_female_brai...
5,F,"(C4) 'control_male_VNC - control_male_brain', ..."
6,G,(C4) 'control_male_VNC - control_female_brain'...
7,H,"(C4) 'control_male_VNC - control_female_VNC', ..."
8,I,"(C4) 'SI_female_brain - control_male_brain', full"
9,J,"(C4) 'SI_female_brain - control_female_brain',..."


In [4]:
tissue_contrasts = comparison_key[
    comparison_key["Comparison"].str.contains(r"brain|VNC", case=False, regex=True)
].copy()
tissue_contrasts["sex_stratified"] = tissue_contrasts["Comparison"].str.contains(
    r"female|male", case=False, regex=True
)
pooled_tissue_contrasts = tissue_contrasts.loc[~tissue_contrasts["sex_stratified"]]

display(tissue_contrasts)
raw_count_sheets = [
    sheet for sheet in workbook.sheet_names
    if "count" in sheet.lower() and sheet.lower() not in {"class sizes"}
]
print(f"Pooled-sex Brain-versus-VNC contrasts in workbook: {len(pooled_tissue_contrasts)}")
print(f"Potential raw-count sheets: {raw_count_sheets or 'none'}")
if pooled_tissue_contrasts.empty:
    print(
        "No pooled-sex Brain-versus-VNC DESeq2 result is present. The displayed tissue contrasts "
        "are stratified by sex; do not combine their fold changes or p-values."
    )
if not raw_count_sheets:
    print(
        "A sex-aggregated DESeq2 reanalysis cannot be fit from these result tables alone. "
        "Provide the raw integer gene-count matrix (genes by sample, sample IDs matching Design) "
        "to fit the tissue effect with sex omitted."
    )

,Key,Comparison,sex_stratified
2,C,(C4) 'control_female_brain - control_male_brai...,True
3,D,(C4) 'control_female_VNC - control_male_brain'...,True
4,E,(C4) 'control_female_VNC - control_female_brai...,True
5,F,"(C4) 'control_male_VNC - control_male_brain', ...",True
6,G,(C4) 'control_male_VNC - control_female_brain'...,True
7,H,"(C4) 'control_male_VNC - control_female_VNC', ...",True
8,I,"(C4) 'SI_female_brain - control_male_brain', full",True
9,J,"(C4) 'SI_female_brain - control_female_brain',...",True
10,K,"(C4) 'SI_female_brain - control_female_VNC', full",True
11,L,"(C4) 'SI_female_brain - control_male_VNC', full",True


Pooled-sex Brain-versus-VNC contrasts in workbook: 0
Potential raw-count sheets: none
No pooled-sex Brain-versus-VNC DESeq2 result is present. The displayed tissue contrasts are stratified by sex; do not combine their fold changes or p-values.
A sex-aggregated DESeq2 reanalysis cannot be fit from these result tables alone. Provide the raw integer gene-count matrix (genes by sample, sample IDs matching Design) to fit the tissue effect with sex omitted.
